# Reference Benchmarks: MMLU, HellaSwag, and Contamination

Standardized benchmarks enabled apples-to-apples model comparisons across organizations. Understanding how they were built, what they measure, and why contamination is such a serious problem is foundational for interpreting published model results.

## The Era of Reference Benchmarks

Before LLMs, NLP benchmarks were task-specific: SQuAD for reading comprehension, CoNLL for NER, WMT for translation. The community needed a unified way to measure general capability.

**MMLU** (Hendrycks et al. 2020): 57 subjects, 15,908 multiple-choice questions spanning STEM, humanities, and professional domains. Designed to test world knowledge and reasoning.

**HellaSwag** (Zellers et al. 2019): sentence completion requiring physical commonsense reasoning. Designed to be adversarially filtered against BERT — humans achieve 95%, early models struggled at 40%.

**ARC** (Clark et al. 2018): grade-school science questions, split into Easy and Challenge sets based on whether retrieval-based systems could answer them.

These benchmarks drove the field for 2018-2023, then began saturating as models approached or exceeded human baselines.

## Few-Shot Prompting Format

Reference benchmarks are typically evaluated with k-shot prompting: provide k example question-answer pairs in the prompt, then ask the model to answer the target question. The standard for MMLU is 5-shot.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random

@dataclass
class Question:
    text: str
    choices: list
    answer: str
    subject: str = 'general'

@dataclass
class BenchResult:
    subject: str
    n_correct: int
    n_total: int
    few_shot: int
    accuracy: float

def fmt_mcq(q: Question, include_answer: bool = False) -> str:
    letters = 'ABCD'
    lines = [q.text]
    for i, choice in enumerate(q.choices):
        lines.append(f'{letters[i]}. {choice}')
    if include_answer:
        lines.append(f'Answer: {q.answer}')
    return '\n'.join(lines)

def run_benchmark(questions: list, model_fn: Callable, few_shot: int = 5,
                  subject: str = 'general', seed: int = 42) -> BenchResult:
    rng = random.Random(seed)
    pool = questions[:]
    correct = 0
    for i, q in enumerate(questions):
        # sample few-shot examples from other questions
        others = [x for j, x in enumerate(pool) if j != i]
        examples = rng.sample(others, min(few_shot, len(others)))
        prompt = ''
        for ex in examples:
            prompt += fmt_mcq(ex, include_answer=True) + '\n\n'
        prompt += fmt_mcq(q, include_answer=False) + '\nAnswer:'
        pred = model_fn(prompt)
        if pred.strip().upper().startswith(q.answer.upper()):
            correct += 1
    acc = correct / len(questions)
    return BenchResult(subject=subject, n_correct=correct, n_total=len(questions),
                       few_shot=few_shot, accuracy=acc)

# Mock dataset (in practice these come from the MMLU dataset files)
sample_questions = [
    Question('What is the powerhouse of the cell?',
             ['Nucleus', 'Mitochondria', 'Ribosome', 'Golgi apparatus'], 'B', 'biology'),
    Question('What year did World War II end?',
             ['1943', '1944', '1945', '1946'], 'C', 'history'),
    Question('What is the derivative of x^2?',
             ['x', '2x', 'x^2', '2x^2'], 'B', 'math'),
    Question('Which element has atomic number 79?',
             ['Silver', 'Platinum', 'Gold', 'Mercury'], 'C', 'chemistry'),
    Question('What is the speed of light in m/s?',
             ['3e6', '3e8', '3e10', '3e12'], 'B', 'physics'),
]

# Mock model: always picks B (simulates a biased system)
def mock_model_b(prompt):
    return 'B'

result = run_benchmark(sample_questions, mock_model_b, few_shot=2)
print(f'Subject: {result.subject}')
print(f'Accuracy: {result.accuracy:.1%} ({result.n_correct}/{result.n_total})')
print(f'Few-shot: {result.few_shot}')

## Contamination: The Core Reliability Problem

Benchmark contamination occurs when test examples appear in a model's training data. Since most LLMs are trained on large internet crawls, benchmark questions that were posted online (in study guides, forums, academic PDFs) are often memorized.

This is a severe problem for the field: a model scoring 90% on MMLU may be mostly recalling memorized answers rather than demonstrating reasoning. MMLU questions appear on numerous education websites.

Contamination detection methods:
1. **n-gram overlap**: check if test question tokens appear verbatim in training data (requires access to training corpus)
2. **Paraphrase sensitivity**: if a model scores much lower on paraphrased versions of the same question, it likely memorized the original
3. **Option shuffling sensitivity**: shuffle answer choices; if accuracy drops dramatically, the model remembered the answer letter not the content
4. **Canary strings**: embed unique identifiers in benchmark data before release; search for them in model completions

In [ ]:
def check_contamination_via_paraphrase(model_fn, original: Question, paraphrases: list) -> dict:
    original_prompt = fmt_mcq(original) + '\nAnswer:'
    orig_pred = model_fn(original_prompt).strip().upper()
    orig_correct = orig_pred.startswith(original.answer.upper())
    para_results = []
    for p in paraphrases:
        para_prompt = fmt_mcq(p) + '\nAnswer:'
        pred = model_fn(para_prompt).strip().upper()
        para_results.append(pred.startswith(p.answer.upper()))
    para_acc = sum(para_results)/len(para_results) if para_results else 0
    return {
        'original_correct': orig_correct,
        'paraphrase_accuracy': para_acc,
        'accuracy_drop': (1.0 if orig_correct else 0.0) - para_acc,
        'contamination_signal': orig_correct and para_acc < 0.5,
    }

# Simulate a 'contaminated' model that knows the exact question
def contaminated_model(prompt):
    if 'powerhouse' in prompt.lower():
        return 'B'  # memorized: Mitochondria is B
    return 'A'     # random guess otherwise

original_q = sample_questions[0]
paraphrase_q = Question(
    'Which organelle is the primary site of ATP production?',
    ['Cell nucleus', 'Mitochondrion', 'Endoplasmic reticulum', 'Lysosome'], 'B', 'biology'
)

result = check_contamination_via_paraphrase(contaminated_model, original_q, [paraphrase_q])
print('Contamination check:')
for k, v in result.items():
    print(f'  {k}: {v}')

## Post-Saturation: What Comes After Benchmarks

By 2024, GPT-4 class models were achieving 85-90% on MMLU, approaching or exceeding estimated human performance. The field responded with harder benchmarks:

**GPQA** (Graduate-Level Google-Proof Q&A): questions that domain experts answer correctly but are not findable via web search.

**FrontierMath**: unpublished math problems from working mathematicians, specifically designed to prevent contamination.

**LiveBench**: dynamically updated with new questions from recent events, ensuring they could not appear in training data.

The general lesson: static benchmarks with fixed question sets have a lifecycle. They are useful until models saturate them, then they measure memorization more than capability. Evaluation infrastructure must continuously refresh.